In [3]:
# Load topic data
import os
import pandas as pd

output_dir = './output'
file_path = os.path.join(output_dir, 'topics.csv')
topics = pd.read_csv(file_path, lineterminator='\n')

topic_names = topics['Name'].tolist()
topics

,Topic,Name,Words\r
0,0,Substance Use Disorder,"alcohol,opioid,substance,drinking,use,dependen..."
1,1,Gut-Brain Axis,"gut,microbiota,microbiome,probiotics,axis,inte..."
2,2,HIV/AIDS Management,"hiv,aids,antiretroviral,living,art,plwh,plhiv,..."
3,3,Suicidal behavior,"suicide,suicidal,ideation,attempts,nssi,attemp..."
4,4,Autism Spectrum Disorder,"autism,asd,autistic,spectrum,children,sensory,..."
...,...,...,...
968,968,Iron homeostasis and neurodegeneration,"iron,ferroptosis,lcn2,overload,labile,tbi,nrf2..."
969,969,Spinal pain management,"epidural,radicular,herniation,steroid,osteocho..."
970,970,Genetic Variants,"cnvs,variants,deletion,schizophrenia,deletions..."
971,971,Child Exposure to Domestic Violence,"domestic,violence,dv,children,exposed,actualiz..."


In [9]:
# Load document to topic map (binary matrix)

output_file_path = os.path.join('./output', 'binary_matrix.csv')
binary_df = pd.read_csv(output_file_path)

# Convert 'PubDate' to datetime
binary_df['PubDate'] = pd.to_datetime(binary_df['PubDate'], errors='coerce')

# Drop any rows where PubDate is NaT (invalid dates)
binary_df = binary_df.dropna(subset=['PubDate'])



In [7]:
# Set up model and DeepInfra API access
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import pipeline
from openai import OpenAI

SYSTEM_MSG = "You are a helpful expert assistant for working with research synthesis topics from the scientific literature in the field of mental health."

modelname = "Qwen/Qwen3-Next-80B-A3B-Instruct" # "meta-llama/Llama-3.3-70B-Instruct"  

client = OpenAI(
        api_key = "ENTER_KEY_HERE",
        base_url="https://api.deepinfra.com/v1/openai",
)
def generateFromPrompt(promptStr,maxTokens=500):
    messages=[
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": promptStr}
    ]
    completion = client.chat.completions.create(
    model=modelname,
    messages=messages)
    response=completion.choices[0].message.content
    return(response)

generateFromPrompt("Hello!")


"Hello! 😊 How can I help you today? Whether you have questions about mental health research, need help synthesizing scientific literature, or just want to chat—I'm here for you!"

In [8]:
# Baseline - with topic hint, but not specific details or abstracts from the topic 

baselines = {}
for i in [19, 127, 141, 102, 21, 247, 1, 101, 312, 157]:  
    # Random 10 trendy topics used in the expert evaluation
    topicn = topic_names[i]
    promptstr = f"Please generate a list of 3-5 current open questions for the topic '{topicn}' where a systematic review would be helpful. Format response in JSON."
    response = generateFromPrompt(promptstr)
    baselines[i]={"Name":topicn,"Response":response}

baselines

{19: {'Name': 'Treatment-Resistant Depression',
  'Response': '{\n  "topic": "Treatment-Resistant Depression",\n  "open_questions": [\n    {\n      "question": "What is the comparative effectiveness of novel neuromodulation techniques (e.g., TMS, ketamine, esketamine, VNS, and DBS) in reducing depressive symptoms and improving functional outcomes in patients with treatment-resistant depression across different symptom profiles and comorbidities?"\n    },\n    {\n      "question": "Among patients with treatment-resistant depression, what are the long-term (≥2 years) outcomes of pharmacological augmentation strategies (e.g., lithium, thyroid hormone, atypical antipsychotics) versus switching to alternative mechanisms of action (e.g., MDMA-assisted therapy, psilocybin), and how do these outcomes vary by biomarker profiles or genetic subtypes?"\n    },\n    {\n      "question": "How do psychosocial interventions (e.g., CBT, DBT, ACT, and behavioral activation) modify treatment response whe

In [24]:
topics.head()


,Topic,Name,Words\r
0,0,Substance Use Disorder,"alcohol,opioid,substance,drinking,use,dependen..."
1,1,Gut-Brain Axis,"gut,microbiota,microbiome,probiotics,axis,inte..."
2,2,HIV/AIDS Management,"hiv,aids,antiretroviral,living,art,plwh,plhiv,..."
3,3,Suicidal behavior,"suicide,suicidal,ideation,attempts,nssi,attemp..."
4,4,Autism Spectrum Disorder,"autism,asd,autistic,spectrum,children,sensory,..."


In [28]:
# Experimental group - with abstracts from the topic 
import random

experimental = {}
for i in [19, 127, 141, 102, 21, 247, 1, 101, 312, 157]:  
    # Random 10 trendy topics used in the expert evaluation
    topicn = topic_names[i]
    topicwords = topics.at[i,'Words\r']
    abstracts = [ a+' '+b for (a,b) in zip(binary_df.query(f"Topic{i}==1 and PubYear>2024")['PaperTitle'].values,binary_df.query(f"Topic{i}==1 and PubYear>2024")['Abstract'].values) ]
    if len(abstracts) > 10: 
        abstracts = random.sample(abstracts, 10)
    promptstr = f"Please extract a list of 3-5 current open questions for the topic '{topicn}' identified by characterstic words '{topicwords}', where a systematic review would be helpful, from the following recent abstracts for this topic: '{abstracts}'. Format response in JSON. "
    response = generateFromPrompt(promptstr)
    experimental[i]={"Name":topicn,"Abstracts":abstracts, "Response":response}

experimental

{19: {'Name': 'Treatment-Resistant Depression',
  'Abstracts': ["Neurophysiological correlates of ketamine-induced dissociative state in bipolar disorder: insights from real-world clinical settings Ketamine, a dissociative compound, shows promise in treating mood disorders, including treatment-resistant depression (TRD) and bipolar disorder (BD). Despite its therapeutic potential, the neurophysiological mechanisms underlying ketamine's effects are not fully understood. This study explored acute neurophysiological changes induced by subanesthetic doses of ketamine in BD patients with depression using electroencephalography (EEG) biomarkers. A cohort of 30 BD (F = 12) inpatients with TRD undergoing ketamine treatment was included in the study. EEG recordings were performed during one of the ketamine infusions with doses ranging from 0.5 to 1 mg/kg, and subjective effects were evaluated using the Clinician-Administered Dissociative States Scale (CADSS). Both rhythmic and arrhythmic featur